In [10]:
%reload_ext autoreload
%autoreload 2

In [11]:
import os
# Set your API key as an environment variable before running.
# For example: $env:OPENROUTER_API_KEY='sk-or-v1-...' in PowerShell

from graph import pddl_lifecycle_graph

sample_requirements = """
We need a warehouse logistics domain. 
There are robots and crates. 
Robots can move between connected locations. 
A robot can pick up a crate at its current location if its hand is empty. 
A robot can drop a crate at its current location.
"""

initial_state = {
    "requirements": sample_requirements,
    "pddl_domain_blueprint": {},
    "generated_domain_pddl": "",
    "verification_errors": []
}



In [12]:

print("🚀 Running Pipeline with OpenRouter Free Tier...\n")

final_state = pddl_lifecycle_graph.invoke(initial_state)

print("="*60)
print("1. RELATIONAL KNOWLEDGE BLUEPRINT (JSON SCHEMA)")
print("="*60)
import json
print(json.dumps(final_state["pddl_domain_blueprint"], indent=2))

🚀 Running Pipeline with OpenRouter Free Tier...

1. RELATIONAL KNOWLEDGE BLUEPRINT (JSON SCHEMA)
{
  "domain_name": "warehouse_logistics",
  "requirements": [
    ":strips",
    ":typing"
  ],
  "types": {
    "robot": null,
    "crate": null,
    "location": null
  },
  "predicates": [
    "(at?r - robot?l - location)",
    "(at?c - crate?l - location)",
    "(connected?l1 - location?l2 - location)",
    "(hand-empty?r - robot)",
    "(holding?r - robot?c - crate)"
  ],
  "actions": [
    {
      "action_name": "move_robot",
      "parameters": {
        "?r": "robot",
        "?from": "location",
        "?to": "location"
      },
      "preconditions": [
        "(at?r?from)",
        "(connected?from?to)"
      ],
      "effects": [
        "(at?r?to)",
        "(not (at?r?from))"
      ]
    },
    {
      "action_name": "pick_up_crate",
      "parameters": {
        "?r": "robot",
        "?c": "crate",
        "?l": "location"
      },
      "preconditions": [
        "(at?r?l)"

In [13]:

print("\n" + "="*60)
print("2. PROGRAMMATICALLY COMPILED PDDL DOMAIN CODE")
print("="*60)
print(final_state["generated_domain_pddl"])


2. PROGRAMMATICALLY COMPILED PDDL DOMAIN CODE
(define (domain warehouse_logistics)
  (:requirements :strips :typing)

  (:types
    robot crate location - object
  )

  (:predicates
    (at?r - robot?l - location)
    (at?c - crate?l - location)
    (connected?l1 - location?l2 - location)
    (hand-empty?r - robot)
    (holding?r - robot?c - crate)
  )

  (:action move_robot
    :parameters (?r - robot ?from - location ?to - location)
    :precondition (and (at?r?from) (connected?from?to))
    :effect (and (at?r?to) (not (at?r?from)))
  )

  (:action pick_up_crate
    :parameters (?r - robot ?c - crate ?l - location)
    :precondition (and (at?r?l) (at?c?l) (hand-empty?r))
    :effect (and (holding?r?c) (not (at?c?l)) (not (hand-empty?r)))
  )

  (:action drop_crate
    :parameters (?r - robot ?c - crate ?l - location)
    :precondition (and (at?r?l) (holding?r?c))
    :effect (and (at?c?l) (hand-empty?r) (not (holding?r?c)))
  )

)
